In [1]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 50.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 54.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import glob

def aggregate_nnunet_results(base_path, experiments_config, output_filename="aggregated_results.csv"):
    """
    Aggregates metrics from multiple nnU-Net experiments into a single CSV.
    Fetches Dice metrics from the Global summary file and HD95 metrics from the Casewise summary file.
    """
    
    # --- 1. Define Metric Sources ---
    # Map metrics to their specific source CSVs and column names
    
    # For summary_cross_fold_global_2d.csv
    dice_map = {
        'Dice_Mean': 'dice',
        'Dice_Std':  'std_dice'
    }
    
    # For summary_cross_fold_casewise_2d.csv
    hd95_map = {
        'HD95_Mean': 'hd95',
        'HD95_Std':  'std_hd95'
    }
    
    # We want classes 1, 2, 3, 4
    target_classes = [1, 2, 3, 4]
    
    aggregated_rows = []

    print(f"Starting aggregation for {len(experiments_config)} experiments...\n")

    for exp_id, exp_name in experiments_config:
        row_data = {'experiment_name': exp_name}
        
        # Construct the Base Directory Path for this experiment
        dataset_dir = f"Dataset{exp_id}_MagPhase_{exp_name}"
        trainer_dir = "nnUnet3DCustomTrainer__nnUNetPlans__3d_fullres"
        inference_dir = "inference_results"
        validation_dir = f"{exp_name}_validation_EVAL"
        
        base_exp_path = os.path.join(
            base_path, 
            dataset_dir, 
            trainer_dir, 
            inference_dir, 
            validation_dir
        )
        
        # Define the two file paths
        path_dice = os.path.join(base_exp_path, "summary_cross_fold_global_2d.csv")
        path_hd95 = os.path.join(base_exp_path, "summary_cross_fold_casewise_2d.csv")

        # Helper function to process a single file and update row_data
        def process_metrics_file(csv_path, metric_map, file_label):
            if not os.path.exists(csv_path):
                print(f"[WARNING] {file_label} file not found: {csv_path}")
                return

            try:
                df = pd.read_csv(csv_path)
                
                # Ensure Label column is integer for matching
                if 'Label' in df.columns:
                    df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
                
                for cls in target_classes:
                    # Filter for the specific class
                    class_row = df[df['Label'] == cls]
                    
                    if class_row.empty:
                        # Optional: Print verbose only if needed to avoid clutter
                        # print(f"  [INFO] Class {cls} missing in {file_label} for {exp_name}")
                        continue
                    
                    # Extract specific metrics defined in the map
                    for src_col, prefix in metric_map.items():
                        new_col_name = f"{prefix}_class_{cls}"
                        
                        if src_col in class_row.columns:
                            # Get the value (iloc[0] because we expect 1 row per class)
                            val = class_row.iloc[0][src_col]
                            row_data[new_col_name] = val
                        else:
                            print(f"  [WARN] Column '{src_col}' missing in {file_label} for {exp_name}")
                            
            except Exception as e:
                print(f"[ERROR] Failed processing {file_label} for {exp_name}: {e}")

        # --- Process Dice (Global File) ---
        process_metrics_file(path_dice, dice_map, "Global/Dice")

        # --- Process HD95 (Casewise File) ---
        process_metrics_file(path_hd95, hd95_map, "Casewise/HD95")

        aggregated_rows.append(row_data)
        print(f"[SUCCESS] Processed: {exp_id} - {exp_name}")

    # 2. Create DataFrame from aggregated rows
    result_df = pd.DataFrame(aggregated_rows)

    # 3. Reorder Columns
    final_columns = ['experiment_name']
    for cls in target_classes:
        # The order requested: dice, std_dice, hd95, std_hd95
        final_columns.append(f"dice_class_{cls}")
        final_columns.append(f"std_dice_class_{cls}")
        final_columns.append(f"hd95_class_{cls}")
        final_columns.append(f"std_hd95_class_{cls}")

    # Reindex to enforce column order (adds NaNs if columns are missing)
    result_df = result_df.reindex(columns=final_columns)

    # 4. Save to CSV
    result_df.to_csv(output_filename, index=False)
    print(f"\nAggregation complete. Saved to: {os.path.abspath(output_filename)}")


# ["1708"]="patchsize_"

# ["1709"]="patchsize_192,64,168"

# ["1710"]="patchsize_192,96,160"
# ["1711"]="patchsize_4"

# ["1712"]="patchsize_5"

# ["1713"]="patchsize_5_spatial_aug_2"
# ["1714"]="patchsize_5_spatial_aug_1"


# # ["1715"]="patchsize_5_otsu_masking"
# ["1716"]="patchsize_5_soft_loss_fixed"
# ["1717"]="patchsize_5_soft_loss_2_fixed"


# # ["1718"]="patchsize_5_adamw"



# # ["1719"]="patchsize_5_adamw_otsu"
# ["1720"]="patchsize_5_adamw_phase_prepro"
# ["1721"]="patchsize_5_adamw_mag_prepro"
# ["1722"]="patch_5_adamw_soft_loss_3"


# # ["1723"]="patch_5_adamw_spatial_aug_3"

# ["1724"]="patch_5_adamw_mag_one_channel"

# # ["1725"]="patch_5_adamw_aug_no_rot_z"

# # ["1726"]="patch_5_adamw_aug_5"

# # ["1727"]="patch_5_adamw_aug_6"
# # ["1728"]="patch_5_adamw_aug_7"
# # ["1729"]="patch_5_adamw_aug_8"

# ["1730"]="patch_5_adamw_aug_9"
# ["1731"]="patch_5_adamw_aug_10"
# ["1732"]="patch_5_adamw_aug_11"

# # ["1733"]="patch_5_adamw_aug_12"

# # ["1734"]="patch_5_adamw_aug_13"
# # ["1735"]="patch_5_adamw_aug_14"
# # ["1736"]="patch_5_adamw_aug_15"

# # ["1737"]="patch_5_adamw_aug_16"
# # ["1738"]="patch_5_adamw_aug_17"
# # ["1739"]="patch_5_adamw_aug_18"
# # ["1740"]="patch_5_adamw_aug_19"

# # ["1741"]="patch_5_adamw_aug_20"
# # ["1742"]="patch_5_adamw_aug_21"
# # ["1743"]="patch_5_adamw_aug_22"
# # ["1744"]="patch_5_adamw_aug_23"


BASE_PATH = "/home/ge.polymtl.ca/pahoa/nih_project/datasets/3D_datasets/train_datasets/nnunet_dataset/nnUNet_results" 

# Define your experiments here.
EXPERIMENTS = [
    ("1708", "patchsize_"),
    ("1709", "patchsize_192,64,168"),
    # ("1710", "patchsize_192,96,160"),
    
]

aggregate_nnunet_results(BASE_PATH, EXPERIMENTS)

Starting aggregation for 2 experiments...

class row:    Label  Dice_Mean  Dice_Std  HD95_Mean  HD95_Std
0      1   0.819379  0.115531        NaN       NaN 


 : Dice_Mean  --> Index(['Label', 'Dice_Mean', 'Dice_Std', 'HD95_Mean', 'HD95_Std'], dtype='str')
class row:    Label  Dice_Mean  Dice_Std  HD95_Mean  HD95_Std
0      1   0.819379  0.115531        NaN       NaN 


 : Dice_Std  --> Index(['Label', 'Dice_Mean', 'Dice_Std', 'HD95_Mean', 'HD95_Std'], dtype='str')
class row:    Label  Dice_Mean  Dice_Std  HD95_Mean  HD95_Std
0      1   0.819379  0.115531        NaN       NaN 


 : HD95_Mean  --> Index(['Label', 'Dice_Mean', 'Dice_Std', 'HD95_Mean', 'HD95_Std'], dtype='str')
class row:    Label  Dice_Mean  Dice_Std  HD95_Mean  HD95_Std
0      1   0.819379  0.115531        NaN       NaN 


 : HD95_Std  --> Index(['Label', 'Dice_Mean', 'Dice_Std', 'HD95_Mean', 'HD95_Std'], dtype='str')
class row:    Label  Dice_Mean  Dice_Std  HD95_Mean  HD95_Std
1      2   0.810725  0.100706        NaN 

In [ ]:
ls /home/ge.polymtl.ca/pahoa/nih_project/datasets/3D_datasets/train_datasets/nnunet_dataset/nnUNet_results/Dataset1709_MagPhase_patchsize_192,64,168/nnUnet3DCustomTrainer__nnUNetPlans__3d_fullres/inference_results/patchsize_192,64,168_validation_EVAL/

fold_0/  logs/                               summary_cross_fold_global_3d.csv
fold_1/  summary_cross_fold_casewise_2d.csv  summary_cross_fold_smoothness.csv
fold_2/  summary_cross_fold_casewise_3d.csv
fold_3/  summary_cross_fold_global_2d.csv
